In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Load your uploaded files
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

def extract_features(df):
    df = df.copy()
    # Extract Cabin Side (S vs P) as found in your EDA
    df['CabinSide'] = df['Cabin'].str.split('/').str[-1].fillna('Unknown')
    # Create the IsToddler flag (Age 0-5)
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['IsToddler'] = (df['Age'] <= 5).astype(int)
    # Total Spending (Negative correlation found in Figure 4)
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in spend_cols:
        df[col] = df[col].fillna(0)
    df['TotalSpend'] = df[spend_cols].sum(axis=1)

    # Select features and encode categories
    features = ['Age', 'CryoSleep', 'IsToddler', 'TotalSpend', 'CabinSide']
    df['CryoSleep'] = df['CryoSleep'].fillna(False).astype(int)
    X = pd.get_dummies(df[features], columns=['CabinSide'], drop_first=True)
    return X

# 2. Train the Model
X = extract_features(train)
y = train['Transported'].astype(int)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# 3. Check Results
val_acc = accuracy_score(y_val, model.predict(X_val_scaled))
print(f"Your Logistic Regression Accuracy: {val_acc:.4f}")

Your Logistic Regression Accuracy: 0.7395


/tmp/ipython-input-697/1150101367.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['CryoSleep'] = df['CryoSleep'].fillna(False).astype(int)


In [6]:
from sklearn.preprocessing import PolynomialFeatures

# 1. Create Polynomial and Interaction Features
# 'interaction_only=True' prevents squaring variables, focusing on how they affect each other
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_val_poly = poly.transform(X_val_scaled)

# 2. Train the updated model
poly_model = LogisticRegression(max_iter=1000)
poly_model.fit(X_train_poly, y_train)

# 3. Check the new accuracy
poly_acc = accuracy_score(y_val, poly_model.predict(X_val_poly))
print(f"Week 2 Logistic Regression (with Interactions) Accuracy: {poly_acc:.4f}")

# 4. Compare with Week 1
improvement = poly_acc - 0.7395
print(f"Improvement over last week: {improvement:.4f}")

Week 2 Logistic Regression (with Interactions) Accuracy: 0.7395
Improvement over last week: 0.0000


In [7]:
from sklearn.model_selection import GridSearchCV

# 1. Define a range of 'C' values to test
# Smaller C = Stronger Regularization (prevents overfitting)
# Larger C = Weaker Regularization (fits training data more closely)
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

# 2. Use Grid Search to find the best version of your model
grid_search = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_poly, y_train)

# 3. Report the best score
best_acc = grid_search.best_score_
print(f"Week 3 Best Tuned Accuracy: {best_acc:.4f}")
print(f"Best 'C' parameter found: {grid_search.best_params_}")

Week 3 Best Tuned Accuracy: 0.7305
Best 'C' parameter found: {'C': 0.001}
